# 冲突检测与解决

## 概述

在现代数据管线中,尤其是构建知识图谱的管线中,数据往往来自多个异构来源(例如内部数据库、第三方 API、网页抓取)。数据不一致是不可避免的。

**Semantica 冲突解决模块**(`semantica.conflicts`)提供了一个稳健的框架来管理这些数据不一致。它旨在确保你的下游应用只消费高质量、经过协调的数据。

### 什么算"冲突"?

- 当**同一实体的多条记录**在某个字段上不一致时,就产生了冲突。
- Semantica 通常假定每条记录是一个字典,包含:
  - `id`(或 `entity_id`):被描述实体的稳定标识符
  - 一个或多个属性(例如 `name`、`birth_date`、`department`)
  - `source`:值的来源(数据库、抓取、API、文件)
  - 可选的 `timestamp`:观察到该值的时间
- 该模块具备来源感知能力:它能记录**哪些来源贡献了哪些值**,然后用投票或可信度等策略来解决冲突。

### 实用的 API 说明(避免常见的不匹配)

- 想检查单个字段时,使用 `ConflictDetector.detect_value_conflicts(entities, property_name=...)`。
- 想做更广泛的扫描(值/类型/时间等)时,使用 `ConflictDetector.detect_conflicts(entities)`。
- 如果你从知识图谱字典(通过 `GraphBuilder` 构建)开始,请通过 `kg.get("entities", [])` 获取实体。

### 关键能力

1.  **多维度冲突检测**
    *   **值冲突**:同一属性有不同的值(例如 `"Google"` vs `"Google Inc."`)。
    *   **类型冲突**:数据类型不匹配(例如字符串 vs 整数)。
    *   **时间冲突**:时间顺序上的不一致(例如 `start_date` 晚于 `end_date`)。

2.  **溯源与来源追踪**
    *   **细粒度追踪**:将每个属性值追溯到其具体的源文档、页面或 API 调用。
    *   **可信度评分**:为来源分配信任分数(例如内部 HR 数据库为 `0.95`,网页抓取为 `0.60`)。

3.  **自动化解决策略**
    *   **投票**:少数服从多数(适用于多个权重相等的来源)。
    *   **可信度加权**:来自高信任来源的值会覆盖其他值。
    *   **时效性**:最新的数据点胜出。
    *   **专家审核**:将复杂的冲突标记出来,交由人工介入。

4.  **调查与审计**
    *   **调查指南**:为人工分析师自动生成分步指南,以解决棘手的冲突。
    *   **审计轨迹**:记录每个冲突是如何被解决的,以满足合规要求。

## 安装

请确保你的环境中已安装 Semantica:

```bash
pip install semantica
```

- 如果你在 Semantica 仓库内运行本 Notebook,建议使用可编辑安装(这样代码改动会立即生效):
  - `pip install -e .`
- 如果你使用的是托管的 Notebook 环境,`%pip install semantica` 通常比 `!pip install ...` 更可靠,因为它会安装到当前活动的内核中。

In [ ]:
!pip install -q semantica 

In [ ]:
import json
from datetime import datetime

## 步骤 1:模拟多源数据

为了演示该框架,我们将模拟一个涉及员工数据的真实场景。

**场景:**
我们从三个不同的来源收到了 **员工 001** 的记录:

- **HR 数据库**:高度可信的内部来源。
- **LinkedIn 抓取**:可靠性较低的外部来源。
- **公共目录**:过时的公共 API。

**记录的结构(Semantica 所期望的):**

- 每条记录都是描述同一实体(`id`:`emp_001`)的字典。
- 每条记录都包含一个 `source` 键,以便将冲突归因到来源。
- `timestamp` 让你可以应用基于时间的解决策略(例如最新的胜出)。

**冲突:**
*   **`birth_date`**:公共目录列出了不同的年份。
*   **`department`**:LinkedIn 使用了更具体的名称("Software Engineering"),而 HR 数据库中是笼统的 "Engineering"。

In [ ]:
# 1. Define source metadata
sources_metadata = {
    "hr_db": {"credibility": 0.95, "type": "internal_database"},
    "linkedin_scrape": {"credibility": 0.60, "type": "web_scrape"},
    "public_dir": {"credibility": 0.40, "type": "public_api"}
}

# 2. Define entity records from these sources
entity_records = [
    {
        "id": "emp_001",
        "name": "John Doe",
        "birth_date": "1980-05-15",
        "department": "Engineering",
        "source": "hr_db",
        "timestamp": "2023-01-01T10:00:00"
    },
    {
        "id": "emp_001",
        "name": "Jonathan Doe",
        "birth_date": "1980-05-15",
        "department": "Software Engineering",
        "source": "linkedin_scrape",
        "timestamp": "2023-06-15T14:30:00"
    },
    {
        "id": "emp_001",
        "name": "John Doe",
        "birth_date": "1982-05-15",  # Conflict: Different year
        "department": "Engineering",
        "source": "public_dir",
        "timestamp": "2022-12-01T09:00:00"
    }
]

print(f"Loaded {len(entity_records)} records for Employee 001")

## 步骤 2:注册与追踪来源

在我们能够基于信任度有效地解决冲突之前,必须先把来源注册到 `SourceTracker` 中。

`SourceTracker` 充当以下内容的中央注册表:

- **可信度分数**:你对该来源的信任程度。
- **元数据**:有用的上下文(例如来源类型、记录系统 vs 抓取)。

**如何看待可信度分数:**

- 把分数用作*相对排序*(具体的小数并不重要,排名才重要)。
- 从简单开始:`internal_db > vendor_api > web_scrape`。
- 之后可借助分析(例如"哪些来源经常出错?")重新审视这些分数。

我们遍历模拟的来源并注册它们。

In [ ]:
from semantica.conflicts import SourceTracker

source_tracker = SourceTracker()

print("Registering sources...")
for source_id, metadata in sources_metadata.items():
    source_tracker.register_source(
        source_id=source_id,
        source_type=metadata["type"],
        credibility_score=metadata["credibility"]
    )
    print(f"  - Registered '{source_id}' with credibility {metadata['credibility']}")

## 步骤 3:检测冲突

我们使用 `ConflictDetector` 扫描记录中的不一致。

该检测器很灵活,可以配置为检查:

- **特定属性**:只检查 `birth_date` 等关键字段。
- **整个实体的扫描**:扫描某个实体的多个(或全部)属性。

**你会得到什么:**

- 一个 `Conflict` 对象的列表。
- 你通常会检查的有用字段:
  - `conflict_type`(例如 `value_conflict`)
  - `entity_id`、`property_name`
  - `conflicting_values` 和 `sources`
  - `severity` 和 `confidence`

这里,我们显式地检查 `birth_date` 和 `department`。

In [ ]:
from semantica.conflicts import ConflictDetector

# Initialize detector with our populated source tracker
detector = ConflictDetector(source_tracker=source_tracker)

conflicts = []

# 1. Check birth_date
dob_conflicts = detector.detect_value_conflicts(entity_records, "birth_date")
conflicts.extend(dob_conflicts)

# 2. Check department
dept_conflicts = detector.detect_value_conflicts(entity_records, "department")
conflicts.extend(dept_conflicts)

print(f"Detected {len(conflicts)} conflicts:")
for conflict in conflicts:
    print(f"- {conflict.conflict_type.value}: {conflict.property_name} for {conflict.entity_id}")
    print(f"  Values: {conflict.conflicting_values}")
    print(f"  Severity: {conflict.severity}")
    print("--- ")

## 步骤 4:分析冲突模式

处理大型数据集时,单个冲突不如系统性的模式重要。`ConflictAnalyzer` 可以帮助回答以下问题:

- "是否某个特定来源造成了大部分冲突?"
- "冲突是否集中在某个特定的实体类型或属性上?"
- "冲突严重程度和冲突类型的分布如何?"

**如何在管线中使用:**

- 运行分析以识别噪声较大的来源。
- 利用结果调整可信度分数(步骤 2),或优化接入/清洗规则。
- 随时间追踪趋势,以便发现上游系统的回退问题。

In [ ]:
from semantica.conflicts import ConflictAnalyzer

analyzer = ConflictAnalyzer()
analysis = analyzer.analyze_conflicts(conflicts)

print("Conflict Analysis Summary:")
print(f"Total Conflicts: {analysis['total_conflicts']}")
print(f"By Type: {analysis.get('by_type', {}).get('counts')}")
print(f"By Severity: {analysis.get('by_severity', {}).get('counts')}")

## 步骤 5:解决冲突

这是决定信任哪个值的关键步骤。Semantica 提供了灵活的解决策略。

### 策略 A:投票(少数服从多数)
该策略选取出现次数最多的值。它很简单,但把所有来源视为平等。

- 当你有许多质量相近的独立来源时效果最好。
- 如果你有一个应当始终占主导的单一记录系统,则不太合适。

### 策略 B:可信度加权
该策略根据每个值所属来源的 `credibility`(可信度)计算加权分数。

**示例:**
*   `hr_db`(0.95)说 "1980-05-15"
*   `public_dir`(0.40)说 "1982-05-15"

即使多个低质量来源一致认同了错误的日期,高可信度的来源也很可能胜出。

**解决器会返回什么:**

- 一个解决结果的列表,其中每一项通常包括:
  - 是否已解决
  - 选定的值(`resolved_value`)
  - 一个置信度分数
  - 元数据(如属性名),用于支持审计轨迹

运行下一个单元格,比较投票与可信度加权的结果。

In [ ]:
from semantica.conflicts import ConflictResolver

resolver = ConflictResolver()

# CRITICAL: Link the source tracker to the resolver.
# This allows the resolver to look up the credibility scores we registered in Step 2.
resolver.set_source_tracker(source_tracker)

print("--- Resolution: Voting ---")
voting_results = resolver.resolve_conflicts(conflicts, strategy="voting")
for res in voting_results:
    print(f"Property: {res.metadata.get('property_name'):<15} | Resolved Value: {res.resolved_value}")

print("\n--- Resolution: Credibility Weighted ---")
# Notice how the HR DB's value is preferred due to higher credibility
credibility_results = resolver.resolve_conflicts(conflicts, strategy="credibility_weighted")
for res in credibility_results:
    print(f"Property: {res.metadata.get('property_name'):<15} | Resolved Value: {res.resolved_value} (Confidence: {res.confidence:.2f})")

## 步骤 6:生成调查指南

并非所有冲突都能自动解决。高风险或低置信度的解决结果需要人工审核。

`InvestigationGuideGenerator` 会为分析师生成结构化的"飞行计划",详细说明:

- **什么**发生了冲突(实体 + 字段 + 相互竞争的值)。
- **谁**参与其中(哪些来源产生了哪些值)。
- **如何**核实正确的数据(分析师可遵循的可操作步骤)。

**何时生成指南:**

- 低置信度的解决结果。
- 关键字段上的冲突(身份、法定名称、合规属性)。
- 任何你想在写回图之前设置一个人工介入检查点的时候。

In [ ]:
from semantica.conflicts import InvestigationGuideGenerator

guide_generator = InvestigationGuideGenerator()

# Generate a guide for the first conflict (birth_date)
guide = guide_generator.generate_guide(conflicts[0])

print(f"=== {guide.title} ===")
print(f"Summary: {guide.conflict_summary}\n")

print("Investigation Steps:")
for i, step in enumerate(guide.investigation_steps, 1):
    print(f"{i}. {step.description}")
    print(f"   Action: {step.action}")

print("\nRecommended Actions:")
for action in guide.recommended_actions:
    print(f"[ ] {action}")

## 结论

你已经使用 Semantica 成功构建了一条冲突解决管线!

**我们完成的内容回顾:**

- **模拟**了带有真实分歧的多源实体记录。
- **注册**了带可信度分数的来源,建立起信任层级。
- **检测**了特定属性上的值冲突。
- **分析**了冲突,以了解按类型和严重程度的分布。
- **解决**了冲突,使用了投票和可信度加权策略。
- **生成**了调查指南,以支持人工审核。

**在真实项目中建议的下一步:**

- 与你的接入层集成,使每个抽取出的值都包含 `source` 和(理想情况下)`timestamp`。
- 使用 `ConflictDetector.detect_conflicts(...)` 将检测范围扩展到值冲突之外。
- 存储解决结果和指南输出,为下游使用者构建审计轨迹。